In [1]:
import sys
sys.path.append('..')

from Agent.config import LLM, SMALL_LLM, SMART_LLM, CLIENT

### 파일 시스템 백앤드

In [4]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends.filesystem import FilesystemBackend

# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=LLM,
    backend=FilesystemBackend(root_dir="/Users/user/{project}", virtual_mode=True),
    skills=["/Users/user/{project}/skills/"],
    interrupt_on={
        "write_file": True,  # Default: approve, edit, reject
        "read_file": False,  # No interrupts needed
        "edit_file": True    # Default: approve, edit, reject
    },
    checkpointer=checkpointer,  # Required!
)

In [8]:
import os

project_dir = "C:/Study/DeepAgent"  # 실제 경로로 변경
os.makedirs(f"{project_dir}/skills", exist_ok=True)
os.makedirs(f"{project_dir}/data", exist_ok=True)

# 테스트용 파일 하나 만들어두기
# CP949로 읽어서 UTF-8로 다시 저장
with open(f"{project_dir}/data/sample.txt", "w", encoding="utf-8") as f:
    f.write("매출: 1000만원\n비용: 700만원\n영업이익: 300만원")

In [9]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends.filesystem import FilesystemBackend

agent = create_deep_agent(
    backend=FilesystemBackend(root_dir=project_dir, virtual_mode=True),
    checkpointer=MemorySaver(),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "data 폴더에 어떤 파일이 있는지 확인하고, sample.txt 내용을 분석해줘"}]},
    config={"configurable": {"thread_id": "test-1"}},
)

print(result["messages"][-1].content)

## 분석 결과

**data 폴더 내 파일:**
- `sample.txt` (1개)

**sample.txt 내용 분석:**

파일에는 간단한 재무 정보가 포함되어 있습니다:

- **매출**: 1,000만원
- **비용**: 700만원
- **영업이익**: 300만원

수치 검증:
- 매출 - 비용 = 1,000만원 - 700만원 = 300만원 ✓
- 영업이익률: 30% (300만원 / 1,000만원)

재무 상태가 건전하며, 매출 대비 30%의 영업이익률을 보이고 있습니다.


In [10]:
result

{'messages': [HumanMessage(content='data 폴더에 어떤 파일이 있는지 확인하고, sample.txt 내용을 분석해줘', additional_kwargs={}, response_metadata={}, id='94b4b99f-f0fa-4cef-8f19-0fcf34fbe437'),
  AIMessage(content=[{'id': 'toolu_016X3nSWy2VBh1HjnHpi8YbQ', 'caller': {'type': 'direct'}, 'input': {'path': '/data'}, 'name': 'ls', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_017xP3FweaohvjRrmY7MkmaG', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 6551, 'inference_geo': 'not_available', 'input_tokens': 3, 'output_tokens': 52, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c98ad-8f34-7490-98b4-cf74dcac8618-0', tool_calls=[{'name': 'ls', 'args': {'path': '/data'}, 'id': 'toolu_

In [11]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": 
        "sample.txt의 재무 데이터를 분석해서 report.md로 저장해줘"}]},
    config={"configurable": {"thread_id": "test-2"}},
)

print(result["messages"][-1].content)

재무 데이터 분석을 완료하고 `/report.md` 파일로 저장했습니다.

**분석 내용:**
- 매출 1,000만원, 비용 700만원, 영업이익 300만원
- 영업이익률: 30% (양호한 수준)
- 비용률: 70%
- 수익성은 안정적이나, 비용 효율화를 통한 개선 여지 있음


### Human-in-the-loop

In [17]:
import os
from langchain.tools import tool
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends.filesystem import FilesystemBackend

# 1. 프로젝트 폴더 구조 생성
project_dir = "C:/Study/DeepAgent"
os.makedirs(f"{project_dir}/data", exist_ok=True)

# 테스트용 파일 생성
with open(f"{project_dir}/data/sample.txt", "w", encoding="utf-8") as f:
    f.write("매출: 1000만원\n비용: 700만원\n영업이익: 300만원")

with open(f"{project_dir}/data/old_report.txt", "w", encoding="utf-8") as f:
    f.write("2024년 1분기 보고서 - 더 이상 필요 없는 파일")


# 2. 커스텀 도구 정의
@tool
def delete_file(path: str) -> str:
    """Delete a file from the filesystem."""
    full_path = os.path.join(project_dir, path.lstrip("/"))
    if os.path.exists(full_path):
        os.remove(full_path)
        return f"✅ 삭제 완료: {path}"
    return f"❌ 파일 없음: {path}"

@tool
def read_file(path: str) -> str:
    """Read a file from the filesystem."""
    full_path = os.path.join(project_dir, path.lstrip("/"))
    if os.path.exists(full_path):
        with open(full_path, "r", encoding="utf-8") as f:
            return f.read()
    return f"❌ 파일 없음: {path}"


# 3. 에이전트 생성 (FilesystemBackend + Human-in-the-Loop)
checkpointer = MemorySaver()

agent = create_deep_agent(
    model="claude-sonnet-4-5-20250929",
    backend=FilesystemBackend(root_dir=project_dir, virtual_mode=True),
    tools=[delete_file, read_file],  # read_file, ls 등은 FilesystemBackend가 자동 제공
    interrupt_on={
        "delete_file": True,   # 삭제 전에 반드시 승인
        "read_file": False,    # 읽기는 자유
        "write_file": False,   # 쓰기도 자유 (테스트 편의상)
    },
    checkpointer=checkpointer,
)

print("에이전트 생성 완료!\n")

에이전트 생성 완료!



In [18]:
# 4. 체험 시나리오: 파일 분석 → 보고서 저장 → 이메일 발송 → 구 파일 삭제

thread_config = {"configurable": {"thread_id": "hitl-test-1"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": 
        "data/sample.txt를 분석해서 report.md로 저장해줘. "
        "마지막으로 data/old_report.txt는 삭제해줘."
    }]},
    config=thread_config,
)

# 5. 에이전트가 어디서 멈췄는지 확인
last_msg = result["messages"][-1]
print("=== 에이전트가 멈춤 (승인 대기 중) ===")
if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"  도구: {tc['name']}")
        print(f"  인자: {tc['args']}")
print()

=== 에이전트가 멈춤 (승인 대기 중) ===
  도구: write_file
  인자: {'file_path': '/report.md', 'content': '# 재무 분석 보고서\n\n## 데이터 분석 결과\n\n### 재무 현황\n- **매출**: 1,000만원\n- **비용**: 700만원\n- **영업이익**: 300만원\n\n### 수익성 분석\n- **영업이익률**: 30% (영업이익 300만원 / 매출 1,000만원)\n- **비용 비율**: 70% (비용 700만원 / 매출 1,000만원)\n\n### 종합 평가\n영업이익률 30%는 양호한 수준으로, 매출 대비 비용이 효율적으로 관리되고 있습니다. \n수익성이 안정적이며, 지속적인 성장 가능성이 있는 것으로 보입니다.'}
  도구: delete_file
  인자: {'path': '/data/old_report.txt'}



In [19]:
from langgraph.types import Command

# 승인
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=thread_config,
)

# # 거부
# result = agent.invoke(
#     Command(resume={"decisions": [{"type": "reject"}]}),
#     config=thread_config,
# )

# # 수정 (인자를 바꿔서 실행)
# result = agent.invoke(
#     Command(resume={"decisions": [{
#         "type": "edit",
#         "edited_action": {
#             "name": "send_email",
#             "args": {"to": "other@company.com", "subject": "수정", "body": "수정된 내용"}
#         }
#     }]}),
#     config=thread_config,
# )

### Skills 시험

In [2]:
import os
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends.filesystem import FilesystemBackend

project_dir = "C:/Study/DeepAgent"

agent = create_deep_agent(
    backend=FilesystemBackend(root_dir=project_dir, virtual_mode=True),
    skills=["/skills/"],
    checkpointer=MemorySaver(),
)

In [ ]:
# 스킬 발동: 프로젝트 폴더 정리 요청
result = agent.invoke(
    {"messages": [{"role": "user", "content": 
        "file-organizer 스킬을 사용해서 현재 프로젝트 폴더 구조를 분석하고, 개선점을 제안해줘"}]},
    config={"configurable": {"thread_id": "organizer-test-1", "recursion_limit": 10}},
)

# 도구 호출 추적
for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[도구 호출] {tc['name']}({tc['args']})")

print("\n=== 최종 응답 ===")
print(result["messages"][-1].content)

### 메모리 -> md 파일 이용한 메모리

In [33]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends import FilesystemBackend

# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()

agent = create_deep_agent(
    backend=FilesystemBackend(root_dir="C:/Study/DeepAgent", virtual_mode=True),
    memory=["/AGENTS.md"],
    checkpointer=checkpointer,  # Required!
)

In [34]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "너 이름은 이제 춘식이여 잘 기억하도록 해"}]},
    config={"configurable": {"thread_id": "memory-test-1"}},
)

In [35]:
result

{'messages': [HumanMessage(content='너 이름은 이제 춘식이여 잘 기억하도록 해', additional_kwargs={}, response_metadata={}, id='7552a378-a852-4219-8b06-0cf7bce3d95b'),
  AIMessage(content=[{'id': 'toolu_01KyxED69zGSFf5GmJ6gK54D', 'caller': {'type': 'direct'}, 'input': {'file_path': '/AGENTS.md'}, 'name': 'read_file', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01N15yJ6G6grco2kwLHjZzEt', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 7640}, 'cache_creation_input_tokens': 7640, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 3, 'output_tokens': 59, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c98f0-a380-7c13-a5f2-a20b26b97108-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/AGENTS.md'}

### Structured Output

In [ ]:
import os
from typing import Literal
from pydantic import BaseModel, Field
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient()

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

class WeatherReport(BaseModel):
    """A structured weather report with current conditions and forecast."""
    location: str = Field(description="The location for this weather report")
    temperature: float = Field(description="Current temperature in Celsius")
    condition: str = Field(description="Current weather condition (e.g., sunny, cloudy, rainy)")
    humidity: int = Field(description="Humidity percentage")
    wind_speed: float = Field(description="Wind speed in km/h")
    forecast: str = Field(description="Brief forecast for the next 24 hours")


agent = create_deep_agent(
    model=LLM,
    response_format=WeatherReport,          ## 여기서 응답 형식을 WeatherReport로 지정
    tools=[internet_search]
)

In [38]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What's the weather like in seoul?"
    }]
})

print(result["structured_response"])
# location='San Francisco, California' temperature=18.3 condition='Sunny' humidity=48 wind_speed=7.6 forecast='Pleasant sunny conditions expected to continue with temperatures around 64°F (18°C) during the day, dropping to around 52°F (11°C) at night. Clear skies with minimal precipitation expected.'

location='Seoul, South Korea' temperature=8.0 condition='Overcast' humidity=61 wind_speed=3.6 forecast='Cool temperatures continuing with cloudy skies. Temperatures expected to range between 0-8°C over the next 24 hours.'


In [39]:
result

{'messages': [HumanMessage(content="What's the weather like in seoul?", additional_kwargs={}, response_metadata={}, id='e62a2e5c-64e8-4fa6-aa42-1afedbbeabfe'),
  AIMessage(content=[{'id': 'toolu_01WFjY9eq2A9DeEZ2DXJVbLk', 'caller': {'type': 'direct'}, 'input': {'query': 'Seoul weather current conditions forecast', 'max_results': 3}, 'name': 'internet_search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01KB2hrCRZYdcBuBeZLo3gkx', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 6915}, 'cache_creation_input_tokens': 6915, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 19, 'output_tokens': 61, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c9998-a162-7272-b374-b788d0f6dc3a-0', tool_ca